# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 02.04 · Consolidación y validación humana

Consolida por precedencia y sirve una interfaz sin datos masivos incrustados.

El acuerdo entre codificadores requiere definir unidad, categorías y medida, no solo contar coincidencias [1]. La intervención humana tampoco garantiza por sí sola calidad en tareas subjetivas asistidas por LLM [2], y mostrar primero la sugerencia puede producir influencia o anclaje [3]. Mostrarla desde el inicio, usar diálogos compactos y permitir lotes confirmados por video/canal son decisiones operativas; la precedencia humana, la adjudicación y el guardado *append-only* permanecen obligatorios.

**Contrato de etiquetas v2.1:** cinco salidas entrenadas: `SEGURO`, `RACISMO_DISCRIMINACION`, `ATAQUE_POR_GENERO_IDENTIDAD`, `ACOSO_AMENAZA` y `CONTENIDO_SEXUAL`. `SEGURO` es excluyente; las cuatro categorías de daño son multietiqueta y pueden coexistir. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

In [4]:
from pathlib import Path
import sys

def find_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('No se encontró pyproject.toml')

ROOT = find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from moderacion_peru.notebook_ui import show_callout, show_command, show_result, show_summary, show_table
OPERATIONAL_PROMPT=ROOT/'config/prompt_operacional_ollama_v3_2.md'
if not OPERATIONAL_PROMPT.is_file():
    raise FileNotFoundError(f'Falta el prompt operacional vigente: {OPERATIONAL_PROMPT}')
show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local', 'prompt_operacional': OPERATIONAL_PROMPT}, tone='success')


raíz,D:/trabajo_PLN/Trabajo_PLN-MIA-Grupo4
backend,local
prompt_operacional,D:/trabajo_PLN/Trabajo_PLN-MIA-Grupo4/config/prompt_operacional_ollama_v3_2.md


## Consolidación

In [5]:
from tqdm.auto import tqdm
from moderacion_peru.consolidation import consolidate_annotations
SOURCES=[p for p in [ROOT/'datos/etiquetado/cascada_deepseek_v4/primary_flash_v3_2.jsonl',ROOT/'datos/etiquetado/cascada_deepseek_v4/review_pro_v3_2.jsonl',ROOT/'datos/etiquetado/cascada_qwen_hf/qwen3_4b_review_v3_2.jsonl',ROOT/'datos/etiquetado/cascada_qwen_hf/qwen3_1_7b_primary_v3_2.jsonl'] if p.exists()]
CHUNKS=ROOT/'datos/processed/chunks_v2.jsonl'
TRANSCRIPTS=ROOT/'datos/raw/transcripts_raw.jsonl'
OUTPUT=ROOT/'datos/etiquetado/consolidado/anotaciones_v2.jsonl'
consolidation_progress={'bar':None}
CONSOLIDATION_PHASES={'loading_annotations':'Leyendo campañas','loading_chunks':'Cargando chunks','loading_transcripts':'Cargando transcripciones','consolidating':'Consolidando propuestas','checking_existing':'Verificando salida existente'}
def report_consolidation_progress(event):
    if event['status']=='phase_started':
        if consolidation_progress.get('bar') is not None:
            consolidation_progress['bar'].close()
        consolidation_progress['bar']=tqdm(total=event.get('total'),desc=CONSOLIDATION_PHASES.get(event['phase'],event['phase']),unit='registro')
        return
    bar=consolidation_progress.get('bar')
    if bar is not None and event.get('advance'):
        bar.update(event['advance'])
        if 'conflicts' in event:
            bar.set_postfix(conflictos=event['conflicts'])
    if event['status']=='finished' and bar is not None:
        bar.close()
        consolidation_progress['bar']=None

if SOURCES:
    try:
        consolidation_result=consolidate_annotations(SOURCES,OUTPUT,chunks_source=CHUNKS,transcripts_source=TRANSCRIPTS,progress_callback=report_consolidation_progress)
    finally:
        if consolidation_progress.get('bar') is not None:
            consolidation_progress['bar'].close()
    show_result('Consolidación de campañas',consolidation_result,tone='success')
else:
    show_callout('Sin campañas','No hay propuestas para consolidar todavía.',tone='warning')

Leyendo campañas: 0registro [00:00, ?registro/s]

Cargando chunks: 0registro [00:00, ?registro/s]

Cargando transcripciones: 0registro [00:00, ?registro/s]

Consolidando propuestas:   0%|          | 0/182461 [00:00<?, ?registro/s]

Verificando salida existente: 0registro [00:00, ?registro/s]

status,noop
chunks,182461
conflicts,0


## Frontend

Ejecute los comandos siguientes en una terminal **PowerShell** abierta en la raíz del repositorio, no dentro de una celda Python. Cada bloque incluye un botón **Copiar**; la celda Python que aparece después solo presenta los comandos y no los ejecuta.

### Preparación inicial

Este bloque solo es necesario la primera vez (o si se eliminó `.venv`):

<div style="margin:8px 0 18px">
<button type="button" onclick="const code=this.nextElementSibling.innerText.trim(); navigator.clipboard.writeText(code); this.textContent='Copiado'; setTimeout(() => { this.textContent='Copiar'; }, 1200);" style="float:right;margin:6px;border:1px solid #94a3b8;background:#fff;border-radius:6px;padding:5px 10px;cursor:pointer;font-weight:600">Copiar</button>
<pre style="clear:both;overflow-x:auto"><code>
py -3.12 -m venv .venv
.\.venv\Scripts\python.exe -m pip install --upgrade pip
.\.venv\Scripts\python.exe -m pip install -e ".[datos,etiquetado,cuadernos,dev]"
</code></pre>
</div>

### Inicio del frontend

<div style="margin:8px 0 18px">
<button type="button" onclick="const code=this.nextElementSibling.innerText.trim(); navigator.clipboard.writeText(code); this.textContent='Copiado'; setTimeout(() => { this.textContent='Copiar'; }, 1200);" style="float:right;margin:6px;border:1px solid #94a3b8;background:#fff;border-radius:6px;padding:5px 10px;cursor:pointer;font-weight:600">Copiar</button>
<pre style="clear:both;overflow-x:auto"><code>
.\.venv\Scripts\modperu.exe serve-labeling `
  --campaign datos/etiquetado/consolidado/anotaciones_v2.jsonl
</code></pre>
</div>

Mantenga esa terminal abierta y visite <http://127.0.0.1:8765>. La interfaz empieza en **Requieren acción**, que contiene únicamente chunks pendientes o diferidos. **Todos los chunks** recorre realmente la campaña completa, incluidos casos resueltos y excluidos. También puede usar **Urgentes**, **Prioritarios Pro** y **Excluidos**; esta última vista permite reclasificar chunks fuera del dataset entrenable.

Para detener el servidor, vuelva a la terminal y presione `Ctrl+C`. En ejecuciones posteriores basta con repetir el bloque de inicio. Si se actualizó el código del servidor, deténgalo y vuelva a iniciar: recargar el navegador por sí solo no activa esos cambios.

In [6]:
setup_command=(
    f'Set-Location "{ROOT}"\n'
    'py -3.12 -m venv .venv\n'
    '.\\.venv\\Scripts\\python.exe -m pip install --upgrade pip\n'
    '.\\.venv\\Scripts\\python.exe -m pip install -e ".[datos,etiquetado,cuadernos,dev]"'
)
start_command=(
    f'Set-Location "{ROOT}"\n'
    f'.\\.venv\\Scripts\\modperu.exe serve-labeling `\n  --campaign "{OUTPUT}"'
)
show_command('Preparación inicial (solo la primera vez)',setup_command,description='Ejecute este bloque en PowerShell si todavía no existe .venv.')
show_command('Iniciar validación humana',start_command,description='Ejecute este bloque en PowerShell y mantenga la terminal abierta.')
show_callout('Abrir o reiniciar el frontend','Visite http://127.0.0.1:8765. Si el servidor ya estaba abierto antes de una actualización, presione Ctrl+C y ejecute otra vez el bloque de inicio; recargar el navegador no actualiza el código Python del servidor.',tone='warning')

## Referencias

[1] R. Artstein and M. Poesio, "Inter-Coder Agreement for Computational Linguistics," Comput. Linguistics, vol. 34, no. 4, pp. 555–596, 2008, doi: 10.1162/coli.07-034-R2.

[2] H. Schroeder, D. Roy, and J. Kabbara, "Just Put a Human in the Loop? Investigating LLM-Assisted Annotation for Subjective Tasks," in Findings ACL, 2025, pp. 25771–25795, doi: 10.18653/v1/2025.findings-acl.1323.

[3] A. S. Choi, S. S. Akter, J. P. Singh, et al., "The LLM Effect: Are Humans Truly Using LLMs, or Are They Being Influenced By Them Instead?" in Proc. EMNLP, 2024, pp. 22032–22054, doi: 10.18653/v1/2024.emnlp-main.1230.